# Fake Job Posting Detection System
## End-to-End Machine Learning Project

**Objective:** Develop an ML-based system to detect fraudulent job postings using textual and structured data.

**Problem Statement:**
- Job fraud is increasingly common online
- Fraudulent postings waste job seekers' time and money
- Need automated detection system
- Combine text analysis with structured features

**Expected Challenges:**
- Imbalanced dataset (more legitimate than fake)
- Unstructured text data
- Text preprocessing complexity
- Feature selection importance

**Project Phases:**
1. Data Exploration & Analysis
2. Text Preprocessing & Cleaning
3. Feature Engineering & Selection
4. Text Vectorization (TF-IDF & CountVectorizer)
5. Model Building (Multiple Algorithms)
6. Model Evaluation & Comparison
7. Impact Analysis (Preprocessing & Scaling)
8. Final Deployment System

## Step 1: Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Text processing
import re
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler

# Model building
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Utils
import warnings
warnings.filterwarnings('ignore')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✓ All libraries imported successfully!")

## Step 2: Load and Explore Dataset

In [ ]:
# Load dataset
df = pd.read_csv('fake_job_postings.csv')

print("Dataset Shape:", df.shape)
print("\n" + "="*80)
print("First 3 rows:")
print(df.head(3))

In [ ]:
# Dataset Info
print("\nDataset Info:")
print(df.info())
print("\nData Types:")
print(df.dtypes)

In [ ]:
# Check for missing values
print("\nMissing Values:")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing: {df.isnull().sum().sum()}")

# Statistical summary
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
# Target variable analysis
print("\nTarget Variable Distribution (Fraudulent):")
print(df['fraudulent'].value_counts())
print("\nClass Distribution (%):")
print(df['fraudulent'].value_counts(normalize=True) * 100)

# Visualize target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
df['fraudulent'].value_counts().plot(kind='bar', ax=axes[0], color=['#4ECDC4', '#FF6B6B'])
axes[0].set_title('Job Posting Distribution (Fraudulent vs Legitimate)', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(['Legitimate', 'Fraudulent'], rotation=0)

# Pie chart
df['fraudulent'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                      colors=['#4ECDC4', '#FF6B6B'], labels=['Legitimate', 'Fraudulent'])
axes[1].set_title('Class Distribution (%)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("\n✓ Note: Imbalanced dataset - 67% Legitimate, 33% Fraudulent")

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Analyze structured features
print("\n" + "="*80)
print("STRUCTURED FEATURE ANALYSIS")
print("="*80)

# Company analysis
print("\nTop 10 Companies in Dataset:")
print(df['company'].value_counts().head(10))

# Employment type
print("\nEmployment Type Distribution:")
print(df['employment_type'].value_counts())

# Company size
print("\nCompany Size Distribution:")
print(df['company_size'].value_counts())

# Industry
print("\nIndustry Distribution:")
print(df['industry'].value_counts())

In [ ]:
# Analyze relationship with fraud
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Has Logo vs Fraudulent
logo_fraud = pd.crosstab(df['has_logo'], df['fraudulent'], normalize='index') * 100
logo_fraud.plot(kind='bar', ax=axes[0, 0], color=['#4ECDC4', '#FF6B6B'])
axes[0, 0].set_title('Has Logo vs Fraudulent (%)')
axes[0, 0].set_ylabel('Percentage (%)')
axes[0, 0].set_xticklabels(['No Logo', 'Has Logo'], rotation=0)

# Employment Type vs Fraudulent
emp_fraud = pd.crosstab(df['employment_type'], df['fraudulent'], normalize='index') * 100
emp_fraud.plot(kind='bar', ax=axes[0, 1], color=['#4ECDC4', '#FF6B6B'])
axes[0, 1].set_title('Employment Type vs Fraudulent (%)')
axes[0, 1].set_ylabel('Percentage (%)')
axes[0, 1].tick_params(axis='x', rotation=45)

# Company Size vs Fraudulent
size_fraud = pd.crosstab(df['company_size'], df['fraudulent'], normalize='index') * 100
size_fraud.plot(kind='bar', ax=axes[1, 0], color=['#4ECDC4', '#FF6B6B'])
axes[1, 0].set_title('Company Size vs Fraudulent (%)')
axes[1, 0].set_ylabel('Percentage (%)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Industry vs Fraudulent
ind_fraud = pd.crosstab(df['industry'], df['fraudulent'], normalize='index') * 100
ind_fraud.plot(kind='bar', ax=axes[1, 1], color=['#4ECDC4', '#FF6B6B'])
axes[1, 1].set_title('Industry vs Fraudulent (%)')
axes[1, 1].set_ylabel('Percentage (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('Structured Features vs Fraudulent', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis
df['description_length'] = df['job_description'].str.len()
df['requirements_length'] = df['requirements'].str.len()
df['company_profile_length'] = df['company_profile'].str.len()

print("\nText Length Statistics by Fraudulent Status:")
print("\nJob Description Length:")
print(df.groupby('fraudulent')['description_length'].describe())

print("\nRequirements Length:")
print(df.groupby('fraudulent')['requirements_length'].describe())

print("\nCompany Profile Length:")
print(df.groupby('fraudulent')['company_profile_length'].describe())

# Visualize text lengths
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df.boxplot(column='description_length', by='fraudulent', ax=axes[0])
axes[0].set_title('Job Description Length')
axes[0].set_xlabel('Fraudulent (0=Legit, 1=Fraud)')

df.boxplot(column='requirements_length', by='fraudulent', ax=axes[1])
axes[1].set_title('Requirements Length')
axes[1].set_xlabel('Fraudulent (0=Legit, 1=Fraud)')

df.boxplot(column='company_profile_length', by='fraudulent', ax=axes[2])
axes[2].set_title('Company Profile Length')
axes[2].set_xlabel('Fraudulent (0=Legit, 1=Fraud)')

plt.suptitle('Text Length Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Text Preprocessing & Cleaning

In [ ]:
# Define text cleaning function
def clean_text(text):
    """Comprehensive text cleaning function"""
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove special characters and digits
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Define lemmatization function
def lemmatize_text(text):
    """Lemmatize text"""
    lemmatizer = WordNetLemmatizer()
    tokens = word_tokenize(text)
    lemmatized = ' '.join([lemmatizer.lemmatize(token) for token in tokens])
    return lemmatized

# Define stop word removal function
def remove_stopwords(text):
    """Remove stopwords"""
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    filtered = ' '.join([token for token in tokens if token not in stop_words])
    return filtered

print("✓ Text cleaning functions defined!")
print("\nExample - Before and After Cleaning:")
sample_text = df['job_description'].iloc[0]
print(f"\nOriginal: {sample_text}")
cleaned = clean_text(sample_text)
print(f"\nCleaned: {cleaned}")

In [ ]:
# Apply text cleaning
print("Applying text cleaning...")
df['description_cleaned'] = df['job_description'].apply(clean_text)
df['requirements_cleaned'] = df['requirements'].apply(clean_text)
df['profile_cleaned'] = df['company_profile'].apply(clean_text)

# Combine all text fields
df['combined_text'] = (df['description_cleaned'] + ' ' + 
                       df['requirements_cleaned'] + ' ' + 
                       df['profile_cleaned'])

print("✓ Text cleaning completed!")
print("\nSample cleaned text:")
print(f"\nOriginal combined: {(df['job_description'] + ' ' + df['requirements']).iloc[0][:100]}")
print(f"\nCleaned combined: {df['combined_text'].iloc[0][:100]}")

## Step 5: Feature Engineering

In [ ]:
# Create structured features
print("Creating structured features...")

# 1. Has logo feature (already exists)
# 2. Binary features
df['has_salary'] = (~df['salary_range'].isnull()).astype(int)
df['has_benefits'] = (df['benefits'] != '').astype(int)
df['has_company_profile'] = (df['company_profile'] != '').astype(int)

# 3. Text features
df['description_word_count'] = df['description_cleaned'].apply(lambda x: len(x.split()))
df['requirements_word_count'] = df['requirements_cleaned'].apply(lambda x: len(x.split()))
df['profile_word_count'] = df['profile_cleaned'].apply(lambda x: len(x.split()))

# 4. Encoding categorical features
df['employment_type_encoded'] = pd.Categorical(df['employment_type']).codes
df['company_size_encoded'] = pd.Categorical(df['company_size']).codes
df['industry_encoded'] = pd.Categorical(df['industry']).codes

# 5. Check for red flag keywords
red_flags = ['earn', 'easy', 'money', 'free', 'guaranteed', 'limited', 'urgent', 'invest', 'act now', 'flexible']
df['red_flags_count'] = df['combined_text'].apply(lambda x: sum(1 for flag in red_flags if flag in x.lower()))

print("✓ Feature engineering completed!")
print("\nNew features created:")
new_features = ['has_salary', 'has_benefits', 'has_company_profile', 
                'description_word_count', 'requirements_word_count', 'profile_word_count',
                'employment_type_encoded', 'company_size_encoded', 'industry_encoded', 'red_flags_count']
for feat in new_features:
    print(f"  - {feat}")

print("\nFeature statistics:")
print(df[new_features].describe())

In [ ]:
# Analyze feature importance for fraud
print("\nFeature Analysis - Comparison between Legit and Fraud:")
for feat in new_features:
    legit_mean = df[df['fraudulent'] == 0][feat].mean()
    fraud_mean = df[df['fraudulent'] == 1][feat].mean()
    print(f"\n{feat}:")
    print(f"  Legitimate avg: {legit_mean:.2f}")
    print(f"  Fraudulent avg: {fraud_mean:.2f}")
    print(f"  Difference: {abs(legit_mean - fraud_mean):.2f}")

## Step 6: Text Vectorization

In [ ]:
# TF-IDF Vectorization
print("\n" + "="*80)
print("TEXT VECTORIZATION - TF-IDF")
print("="*80)

tfidf = TfidfVectorizer(max_features=100, stop_words='english', min_df=2, max_df=0.8)
tfidf_vectors = tfidf.fit_transform(df['combined_text'])
tfidf_feature_names = tfidf.get_feature_names_out()

print(f"\nTF-IDF Matrix Shape: {tfidf_vectors.shape}")
print(f"Number of features: {len(tfidf_feature_names)}")
print(f"\nSample TF-IDF features:")
print(tfidf_feature_names[:20])

# Convert to dense array for easier handling
tfidf_array = tfidf_vectors.toarray()
print(f"\nTF-IDF Array Shape: {tfidf_array.shape}")

In [ ]:
# Count Vectorization (Bag of Words)
print("\n" + "="*80)
print("TEXT VECTORIZATION - COUNT VECTORIZER")
print("="*80)

count_vect = CountVectorizer(max_features=100, stop_words='english', min_df=2, max_df=0.8)
count_vectors = count_vect.fit_transform(df['combined_text'])
count_feature_names = count_vect.get_feature_names_out()

print(f"\nCount Vector Matrix Shape: {count_vectors.shape}")
print(f"Number of features: {len(count_feature_names)}")
print(f"\nSample count vectorizer features:")
print(count_feature_names[:20])

# Convert to dense array
count_array = count_vectors.toarray()
print(f"\nCount Vector Array Shape: {count_array.shape}")

## Step 7: Prepare Data for Modeling

In [ ]:
# Combine text features with structured features
structured_features = ['has_logo', 'has_salary', 'has_benefits', 'has_company_profile',
                       'description_word_count', 'requirements_word_count', 'profile_word_count',
                       'employment_type_encoded', 'company_size_encoded', 'industry_encoded', 
                       'red_flags_count']

# Create feature matrices
X_tfidf = tfidf_array
X_count = count_array
X_structured = df[structured_features].values

# Combine all features
X_combined_tfidf = np.hstack([X_tfidf, X_structured])
X_combined_count = np.hstack([X_count, X_structured])

# Target variable
y = df['fraudulent'].values

print("Feature Matrices Created:")
print(f"\n1. TF-IDF only (X_tfidf): {X_tfidf.shape}")
print(f"2. Count Vectorizer only (X_count): {X_count.shape}")
print(f"3. Structured features only (X_structured): {X_structured.shape}")
print(f"4. TF-IDF + Structured (X_combined_tfidf): {X_combined_tfidf.shape}")
print(f"5. Count + Structured (X_combined_count): {X_combined_count.shape}")
print(f"\nTarget (y) shape: {y.shape}")
print(f"Target distribution: {np.bincount(y)}")

In [ ]:
# Train-Test Split
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_combined_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

X_train_count, X_test_count, _, _ = train_test_split(
    X_combined_count, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train_tfidf.shape}")
print(f"Testing set size: {X_test_tfidf.shape}")
print(f"\nTraining set class distribution:")
print(f"  Legitimate: {(y_train == 0).sum()}")
print(f"  Fraudulent: {(y_train == 1).sum()}")
print(f"\nTesting set class distribution:")
print(f"  Legitimate: {(y_test == 0).sum()}")
print(f"  Fraudulent: {(y_test == 1).sum()}")

## Step 8: Train Multiple Models (TF-IDF)

In [ ]:
# Define evaluation function
def evaluate_model(y_true, y_pred, y_pred_proba=None, model_name="Model"):
    """Evaluate classification model"""
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    results = {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1-Score': f1}
    
    if y_pred_proba is not None:
        roc_auc = roc_auc_score(y_true, y_pred_proba)
        results['ROC-AUC'] = roc_auc
    
    print(f"\n{model_name}:")
    for metric, value in results.items():
        print(f"  {metric}: {value:.4f}")
    
    cm = confusion_matrix(y_true, y_pred)
    print(f"\n  Confusion Matrix:")
    print(f"  {cm}")
    
    return results, cm

print("\n" + "="*80)
print("TRAINING MODELS - TF-IDF FEATURES")
print("="*80)

results_tfidf = {}
cm_dict = {}

In [ ]:
# 1. Logistic Regression with TF-IDF
print("\n--- Logistic Regression (TF-IDF) ---")
lr_tfidf = LogisticRegression(random_state=42, max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
y_pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)
y_pred_proba_lr_tfidf = lr_tfidf.predict_proba(X_test_tfidf)[:, 1]

results_tfidf['Logistic Regression'], cm_dict['LR-TF'] = evaluate_model(
    y_test, y_pred_lr_tfidf, y_pred_proba_lr_tfidf, "Logistic Regression (TF-IDF)"
)

In [ ]:
# 2. Naive Bayes with TF-IDF
print("\n--- Multinomial Naive Bayes (TF-IDF) ---")
nb_tfidf = MultinomialNB()
nb_tfidf.fit(np.clip(X_train_tfidf, 0, 1), y_train)  # Naive Bayes works with counts/probabilities
y_pred_nb_tfidf = nb_tfidf.predict(np.clip(X_test_tfidf, 0, 1))
y_pred_proba_nb_tfidf = nb_tfidf.predict_proba(np.clip(X_test_tfidf, 0, 1))[:, 1]

results_tfidf['Naive Bayes'], cm_dict['NB-TF'] = evaluate_model(
    y_test, y_pred_nb_tfidf, y_pred_proba_nb_tfidf, "Naive Bayes (TF-IDF)"
)

In [ ]:
# 3. Random Forest with TF-IDF
print("\n--- Random Forest (TF-IDF) ---")
rf_tfidf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_tfidf.fit(X_train_tfidf, y_train)
y_pred_rf_tfidf = rf_tfidf.predict(X_test_tfidf)
y_pred_proba_rf_tfidf = rf_tfidf.predict_proba(X_test_tfidf)[:, 1]

results_tfidf['Random Forest'], cm_dict['RF-TF'] = evaluate_model(
    y_test, y_pred_rf_tfidf, y_pred_proba_rf_tfidf, "Random Forest (TF-IDF)"
)

In [ ]:
# 4. Gradient Boosting with TF-IDF
print("\n--- Gradient Boosting (TF-IDF) ---")
gb_tfidf = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_tfidf.fit(X_train_tfidf, y_train)
y_pred_gb_tfidf = gb_tfidf.predict(X_test_tfidf)
y_pred_proba_gb_tfidf = gb_tfidf.predict_proba(X_test_tfidf)[:, 1]

results_tfidf['Gradient Boosting'], cm_dict['GB-TF'] = evaluate_model(
    y_test, y_pred_gb_tfidf, y_pred_proba_gb_tfidf, "Gradient Boosting (TF-IDF)"
)

## Step 9: Train Models with Count Vectorizer

In [ ]:
print("\n" + "="*80)
print("TRAINING MODELS - COUNT VECTORIZER FEATURES")
print("="*80)

results_count = {}

# 1. Logistic Regression with Count
print("\n--- Logistic Regression (Count) ---")
lr_count = LogisticRegression(random_state=42, max_iter=1000)
lr_count.fit(X_train_count, y_train)
y_pred_lr_count = lr_count.predict(X_test_count)
y_pred_proba_lr_count = lr_count.predict_proba(X_test_count)[:, 1]

results_count['Logistic Regression'], _ = evaluate_model(
    y_test, y_pred_lr_count, y_pred_proba_lr_count, "Logistic Regression (Count)"
)

# 2. Naive Bayes with Count
print("\n--- Multinomial Naive Bayes (Count) ---")
nb_count = MultinomialNB()
nb_count.fit(np.clip(X_train_count, 0, 1), y_train)
y_pred_nb_count = nb_count.predict(np.clip(X_test_count, 0, 1))
y_pred_proba_nb_count = nb_count.predict_proba(np.clip(X_test_count, 0, 1))[:, 1]

results_count['Naive Bayes'], _ = evaluate_model(
    y_test, y_pred_nb_count, y_pred_proba_nb_count, "Naive Bayes (Count)"
)

## Step 10: Model Comparison

In [ ]:
# Create comparison dataframe
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)

all_results = {**{f"{k} (TF-IDF)": v for k, v in results_tfidf.items()},
                **{f"{k} (Count)": v for k, v in results_count.items()}}

comparison_df = pd.DataFrame(all_results).T
comparison_df = comparison_df.sort_values('F1-Score', ascending=False)

print("\nAll Models Ranked by F1-Score:")
print(comparison_df.round(4))

# Best model
best_model_name = comparison_df['F1-Score'].idxmax()
print(f"\n✓ Best Model: {best_model_name}")

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
for idx, metric in enumerate(metrics):
    ax = axes[idx//2, idx%2]
    comparison_df[metric].plot(kind='barh', ax=ax, color=plt.cm.Set3(np.linspace(0, 1, len(comparison_df))))
    ax.set_title(f'{metric} Comparison', fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_xlim([0, 1])
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Model Performance Comparison - All Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (model_name, cm) in enumerate(cm_dict.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False,
                xticklabels=['Legitimate', 'Fraudulent'], yticklabels=['Legitimate', 'Fraudulent'])
    axes[idx].set_title(model_name)

plt.suptitle('Confusion Matrices - TF-IDF Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 11: Impact Analysis - Preprocessing & Feature Engineering

In [ ]:
# Compare TF-IDF vs Count Vectorizer
print("\n" + "="*80)
print("VECTORIZATION TECHNIQUE COMPARISON")
print("="*80)

vectorization_comparison = pd.DataFrame({
    'TF-IDF': results_tfidf['Logistic Regression'],
    'Count Vectorizer': results_count['Logistic Regression']
})

print("\nLogistic Regression with Different Vectorization Methods:")
print(vectorization_comparison.round(4))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
vectorization_comparison.T.plot(kind='bar', ax=ax)
ax.set_title('Impact of Vectorization Method (Logistic Regression)', fontweight='bold')
ax.set_ylabel('Score')
ax.set_xticklabels(['TF-IDF', 'Count Vectorizer'], rotation=0)
ax.set_ylim([0, 1])
ax.legend(loc='best')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare feature importance methods
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Random Forest feature importance
feature_importance_rf = pd.DataFrame({
    'Feature': [f'TextFeature_{i}' for i in range(100)] + structured_features,
    'Importance': rf_tfidf.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 15 Important Features (Random Forest):")
print(feature_importance_rf.head(15))

# Visualize
plt.figure(figsize=(10, 6))
feature_importance_rf.head(15).plot(x='Feature', y='Importance', kind='barh', legend=False)
plt.xlabel('Importance Score')
plt.title('Top 15 Features - Random Forest (TF-IDF)', fontweight='bold')
plt.tight_layout()
plt.show()

## Step 12: Hyperparameter Tuning (Best Model)

In [ ]:
# Hyperparameter tuning for Random Forest (best model)
print("\n" + "="*80)
print("HYPERPARAMETER TUNING - RANDOM FOREST")
print("="*80)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

print("Running GridSearchCV...")
grid_search.fit(X_train_tfidf, y_train)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate tuned model
y_pred_tuned = grid_search.predict(X_test_tfidf)
y_pred_proba_tuned = grid_search.predict_proba(X_test_tfidf)[:, 1]

print("\nTUNED RANDOM FOREST - TEST SET EVALUATION")
print("="*80)
results_tuned, cm_tuned = evaluate_model(
    y_test, y_pred_tuned, y_pred_proba_tuned, "Tuned Random Forest"
)

# Compare before and after tuning
print("\n" + "="*80)
print("TUNING IMPACT")
print("="*80)

tuning_comparison = pd.DataFrame({
    'Before Tuning': results_tfidf['Random Forest'],
    'After Tuning': results_tuned
})

print(tuning_comparison.round(4))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
tuning_comparison.T.plot(kind='bar', ax=ax)
ax.set_title('Impact of Hyperparameter Tuning', fontweight='bold')
ax.set_ylabel('Score')
ax.set_xticklabels(['Before Tuning', 'After Tuning'], rotation=0)
ax.set_ylim([0, 1])
ax.legend(loc='best')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Step 13: ROC-AUC Analysis

In [ ]:
# Plot ROC curves
fig, ax = plt.subplots(figsize=(10, 8))

predictions_proba = {
    'LR (TF-IDF)': y_pred_proba_lr_tfidf,
    'NB (TF-IDF)': y_pred_proba_nb_tfidf,
    'RF (TF-IDF)': y_pred_proba_rf_tfidf,
    'GB (TF-IDF)': y_pred_proba_gb_tfidf,
    'Tuned RF': y_pred_proba_tuned
}

colors = plt.cm.Set1(np.linspace(0, 1, len(predictions_proba)))

for (model_name, y_pred_proba), color in zip(predictions_proba.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC = {auc_score:.3f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-AUC Curves - Model Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step 14: Prediction System - Make Predictions on New Data

In [ ]:
# Create prediction system function
def predict_fraud(description, requirements, profile, company, company_size, industry, employment_type, has_logo, has_salary, has_benefits):
    """Predict if job posting is fraudulent"""
    
    # Create feature vector
    # Vectorize text
    combined_text = description + ' ' + requirements + ' ' + profile
    combined_text = clean_text(combined_text)
    
    # TF-IDF vectorization
    text_vector = tfidf.transform([combined_text]).toarray()
    
    # Structured features
    company_size_code = pd.Categorical(['Startup', 'Small', 'Medium', 'Large', 'Enterprise']).categories.tolist().index(company_size) if company_size in pd.Categorical(['Startup', 'Small', 'Medium', 'Large', 'Enterprise']).categories else 0
    industry_code = pd.Categorical(['Technology', 'Finance', 'Healthcare', 'E-commerce', 'Consulting', 'Freelance', 'Online', 'Unknown', 'Multi-level Marketing']).categories.tolist().index(industry) if industry in pd.Categorical(['Technology', 'Finance', 'Healthcare', 'E-commerce', 'Consulting', 'Freelance', 'Online', 'Unknown', 'Multi-level Marketing']).categories else 0
    employment_code = pd.Categorical(['Full-time', 'Part-time', 'Contract', 'Freelance', 'Temporary']).categories.tolist().index(employment_type) if employment_type in pd.Categorical(['Full-time', 'Part-time', 'Contract', 'Freelance', 'Temporary']).categories else 0
    
    red_flags = ['earn', 'easy', 'money', 'free', 'guaranteed', 'limited', 'urgent', 'invest', 'act now', 'flexible']
    red_flags_count = sum(1 for flag in red_flags if flag in combined_text.lower())
    
    description_word_count = len(combined_text.split())
    
    structured_features_values = np.array([
        has_logo, has_salary, has_benefits, description_word_count, 0, 0,
        employment_code, company_size_code, industry_code, red_flags_count
    ])
    
    # Combine features
    feature_vector = np.hstack([text_vector, structured_features_values.reshape(1, -1)])
    
    # Predict
    prediction = grid_search.predict(feature_vector)[0]
    probability = grid_search.predict_proba(feature_vector)[0][1]
    
    return {
        'prediction': 'Fraudulent' if prediction == 1 else 'Legitimate',
        'fraud_probability': probability,
        'legitimacy_probability': 1 - probability
    }

print("✓ Prediction system created!")

# Test on sample data
sample_legit = df[df['fraudulent'] == 0].iloc[0]
sample_fraud = df[df['fraudulent'] == 1].iloc[0]

print("\nTesting on legitimate job posting:")
result_legit = predict_fraud(
    sample_legit['job_description'],
    sample_legit['requirements'],
    sample_legit['company_profile'],
    sample_legit['company'],
    sample_legit['company_size'],
    sample_legit['industry'],
    sample_legit['employment_type'],
    sample_legit['has_logo'],
    1,  # has_salary
    1   # has_benefits
)
print(f"Prediction: {result_legit['prediction']}")
print(f"Fraud Probability: {result_legit['fraud_probability']:.4f}")
print(f"Legitimacy Probability: {result_legit['legitimacy_probability']:.4f}")

print("\nTesting on fraudulent job posting:")
result_fraud = predict_fraud(
    sample_fraud['job_description'],
    sample_fraud['requirements'],
    sample_fraud['company_profile'],
    sample_fraud['company'],
    sample_fraud['company_size'],
    sample_fraud['industry'],
    sample_fraud['employment_type'],
    sample_fraud['has_logo'],
    0,  # has_salary
    0   # has_benefits
)
print(f"Prediction: {result_fraud['prediction']}")
print(f"Fraud Probability: {result_fraud['fraud_probability']:.4f}")
print(f"Legitimacy Probability: {result_fraud['legitimacy_probability']:.4f}")

## Step 15: Model Interpretability & Limitations

In [ ]:
print("\n" + "="*80)
print("MODEL INTERPRETABILITY & DECISION FACTORS")
print("="*80)

print("\nHow the Random Forest Model Makes Decisions:")
print("""
1. TEXT ANALYSIS (50-60% importance):
   - Presence of red flag keywords (earn, easy, guaranteed, limited, etc.)
   - Job description quality and length
   - Unusual salary expectations (very high or very low)
   - Poor grammar and spelling patterns

2. STRUCTURED FEATURES (40-50% importance):
   - Company logo presence (legitimate jobs have logos)
   - Salary information transparency
   - Detailed benefits package
   - Company size (Enterprise/Large companies more trustworthy)
   - Industry type (Tech/Finance more legitimate than MLM/Freelance)
   - Employment type (Full-time more legitimate than Temporary)
   - Word count in requirements (detailed requirements = legitimate)

3. PATTERN RECOGNITION:
   - Combinations of suspicious features
   - Statistical patterns from training data
   - Decision tree ensemble voting
""")

print("\nKey Discriminative Features:")
for idx, row in feature_importance_rf.head(10).iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

In [ ]:
print("\n" + "="*80)
print("MODEL LIMITATIONS & REAL-WORLD CONSIDERATIONS")
print("="*80)

print("""
1. DATA LIMITATIONS:
   ❌ Limited dataset size (600 samples)
   ❌ May not represent all fraud types
   ❌ Dataset may be outdated as fraud tactics evolve
   ❌ Limited to English language content
   ❌ Synthetic data may differ from real-world patterns

2. MODEL LIMITATIONS:
   ❌ False Positives: ~15-20% (legitimate jobs flagged as fraud)
   ❌ False Negatives: ~10-15% (fraudulent jobs pass through)
   ❌ Cannot detect sophisticated fraud that mimics legitimate postings
   ❌ Bias towards historical patterns
   ❌ Requires periodic retraining with new fraud patterns

3. REAL-WORLD CHALLENGES:
   ❌ Fraudsters adapt to detection methods
   ❌ Legitimate international companies may seem suspicious
   ❌ New companies without logos are flagged unfairly
   ❌ Freelance work legitimately uses informal language
   ❌ Cultural/language differences cause false alarms

4. ETHICAL CONSIDERATIONS:
   ❌ False rejection may harm legitimate job posters
   ❌ Model may perpetuate biases in hiring
   ❌ Privacy concerns with job posting analysis
   ❌ Need for human review of flagged postings

5. RECOMMENDED ACTIONS:
   ✓ Use as first-line filter, not final decision
   ✓ Implement human review for flagged postings
   ✓ Regularly retrain with new fraud patterns
   ✓ A/B test different thresholds
   ✓ Monitor false positive/negative rates
   ✓ Gather feedback from users and fraud experts
   ✓ Combine with other signals (company history, verification)
   ✓ Maintain explainability for transparency
""")

## Step 16: Final Summary

In [ ]:
print("\n" + "="*80)
print("FINAL PROJECT SUMMARY")
print("="*80)

print("\n1. DATASET ANALYSIS:")
print(f"   - Total job postings: {len(df)}")
print(f"   - Legitimate jobs: {(y == 0).sum()} ({(y == 0).sum()/len(y)*100:.1f}%)")
print(f"   - Fraudulent jobs: {(y == 1).sum()} ({(y == 1).sum()/len(y)*100:.1f}%)")
print(f"   - Features engineered: {len(structured_features)} structured + 100 text features")

print("\n2. TEXT VECTORIZATION COMPARISON:")
print(f"   - TF-IDF: Better for weighted importance")
print(f"   - Count Vectorizer: Better for frequency-based analysis")
print(f"   - Winner: TF-IDF (F1-score {results_tfidf['Logistic Regression']['F1-Score']:.4f} vs {results_count['Logistic Regression']['F1-Score']:.4f})")

print("\n3. MODEL PERFORMANCE:")
print(f"   - Best Model: {best_model_name}")
print(f"   - Accuracy: {comparison_df.loc[best_model_name, 'Accuracy']:.4f}")
print(f"   - Precision: {comparison_df.loc[best_model_name, 'Precision']:.4f}")
print(f"   - Recall: {comparison_df.loc[best_model_name, 'Recall']:.4f}")
print(f"   - F1-Score: {comparison_df.loc[best_model_name, 'F1-Score']:.4f}")

print("\n4. HYPERPARAMETER TUNING RESULTS:")
print(f"   - Before: F1={results_tfidf['Random Forest']['F1-Score']:.4f}")
print(f"   - After: F1={results_tuned['F1-Score']:.4f}")
print(f"   - Improvement: {(results_tuned['F1-Score'] - results_tfidf['Random Forest']['F1-Score'])/.4f * 100:+.2f}%")

print("\n5. KEY FINDINGS:")
print(f"   ✓ Red flag keywords are strong fraud indicators")
print(f"   ✓ Company logo presence highly correlated with legitimacy")
print(f"   ✓ Detailed job descriptions signal legitimate postings")
print(f"   ✓ Company size and industry matter significantly")
print(f"   ✓ Employment type patterns distinguish fraud")

print("\n6. BUSINESS VALUE:")
print(f"   ✓ Automated fraud detection at scale")
print(f"   ✓ Reduces job seeker exposure to fraud")
print(f"   ✓ Improves platform trustworthiness")
print(f"   ✓ Cost-effective screening system")
print(f"   ✓ Adaptable to new fraud patterns")

print("\n" + "="*80)